In [1]:
import jfr_processor.profilelib.jsonjfr.*

val root = "/home/Martin.Zimen/IdeaProjects/ktor/profiler_output_weekend/"

In [2]:
val jvm_functions = mutableSetOf<String>()
val native_functions = mutableSetOf<String>()
get_samples(root)
    .forEach {
        jvm_functions.addAll(list_functions(it.jvm))
        native_functions.addAll(list_functions(it.native))
    }
val functions = jvm_functions intersect native_functions

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

java.lang.OutOfMemoryError: Required array length 2147483639 + 9 is too large

In [3]:
val naive_profile = JvmNativePair(mutableMapOf<String, Int>(), mutableMapOf<String, Int>())

get_samples(root)
    .forEach {
        fun naive_count(count: MutableMap<String, Int>, samples: Samples) {
            samples.forEach { stackTrace ->
                stackTrace
                    .map { get_name(it) }
                    .toSet()
                    .filter { it in functions }
                    .forEach { count[it] = count.getOrDefault(it, 0) + 1 }
            }
        }
        naive_count(naive_profile.jvm, it.jvm)
        naive_count(naive_profile.native, it.native)
    }


ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

org.jetbrains.kotlinx.jupyter.exceptions.ReplInterruptedException: The execution was interrupted

In [ ]:
%use dataframe
val flist = functions.toList()

val df = dataFrameOf(
    "Function" to flist,
    "Jvm [samples]" to flist.map { naive_profile.jvm[it]!! },
    "Native [samples]" to flist.map { naive_profile.native[it]!! },
)
df.add("Ratio") { (it["Jvm [samples]"] / it["Native [samples]"]) }
df.sortedByDescending("Ratio")

In [17]:
var jvm_filtered = mutableSetOf<StackTrace>()
var native_filtered = mutableSetOf<StackTrace>()

get_samples(root)
    .forEach {
        fun filter_samples(samples: Samples): Set<StackTrace> =
            samples.filter { stackTrace -> stackTrace.map { get_name(it) }.any { it == "kotlin/enums/EnumEntriesList.get" } }.toSet()
        jvm_filtered.addAll(filter_samples(it.jvm))
        native_filtered.addAll(filter_samples(it.native))
    }

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

In [23]:
jvm_filtered.forEach { it.map { get_name(it) }.joinToString(" -- ").let { println(it) } }
println()
native_filtered.forEach { it.map { get_name(it) }.joinToString(" -- ").let { println(it) } }
println()
(native_filtered intersect jvm_filtered).forEach { it.map { get_name(it) }.joinToString(" -- ").let { println(it) } }


kotlin/enums/EnumEntriesList.get -- kotlin/enums/EnumEntriesList.get -- io/ktor/util/date/WeekDay$Companion.from -- io/ktor/util/date/DateJvmKt.toDate -- io/ktor/util/date/DateJvmKt.GMTDate
kotlin/enums/EnumEntriesList.get -- kotlin/enums/EnumEntriesList.get -- io/ktor/util/date/Month$Companion.from -- io/ktor/util/date/DateJvmKt.toDate -- io/ktor/util/date/DateJvmKt.GMTDate
kotlin/enums/EnumEntriesList.get
kotlin/collections/AbstractList$Companion.checkElementIndex$kotlin_stdlib -- kotlin/enums/EnumEntriesList.get -- kotlin/enums/EnumEntriesList.get -- io/ktor/util/date/WeekDay$Companion.from -- io/ktor/util/date/DateJvmKt.toDate
kotlin/enums/EnumEntriesList.get -- kotlin/enums/EnumEntriesList.get -- kotlin/collections/AbstractList$IteratorImpl.next -- io/ktor/util/date/Month$Companion.from -- io/ktor/util/date/GMTDateParser.handleToken
kotlin/collections/AbstractList$Companion.checkElementIndex$kotlin_stdlib -- kotlin/enums/EnumEntriesList.get -- kotlin/enums/EnumEntriesList.get -- i

This approach is wrong for multiple reasons.

* The `--stack-depth` setting
    * if it is set too low, the stdlib frame may be missing.
    * If it is too high, the current processing method runs out of memory (the JSON string alone can be over 5 GB)
* Even if this approach worked, the discrepancy in the number of samples can be caused by other sources than Native/JVM performance difference. Possibly the fork can be much more lower in the stack (and in Ktor it probably is) - in JVM, Java webserver will be used that won't use Kotlin stdlib

This data was originally processed in LibreOffice before I learned there is kotlin DataFrame library. Then I started modifying the code to use DataFrame. While doing that, I realized the whole method is bollocks and stopped midway. Leaving this data analysis non-replicable.